# Study 1 - The GGUF artifact

## Why measure this, and where it runs in production

A GGUF file is the deployment currency of the SLM world: a single binary that packs weights, tokenizer, and architecture metadata so any compatible runtime can serve it. Before we trust its served probabilities, we must know what the file actually promises. The file header says `general.file_type`; the runtime reads that and decides the arithmetic. Two files can look identical and be wildly different inside.

Production question: when you ship a quantized model, what exactly are you shipping? A size? A name? Or a byte layout that silently decides which tensors keep precision and which get coarsened?

## Abstract

We take three SmolLM2 sizes (135M, 360M, 1.7B), convert each from HF safetensors to GGUF v3 F16 with the official converter (vendored at the exact llama.cpp commit that the llama-cpp-python 0.3.34 wheel bundles), then quantize to Q8_0, Q4_K, and Q4_0 with the runtime's own `llama_model_quantize`. We read the file back and report: file sizes, bytes-per-weight (BPW), the `general.file_type` promises, and - critically - how many tensors silently fell back to a coarser type because SmolLM2's inner widths (576, 1536) are not divisible by the k-quant's 256-block.

The artifact is not a black box. It is a contract, and this study reads the fine print.

## Related work, and what changes here

The GGUF format and llama.cpp runtime are the de-facto CPU serving standard. Documentation describes the format and the quant types. What is not documented is the *gap between the name and the math*: the fallback quantization that occurs at build time when a tensor shape is incompatible with the requested block size. That gap is what this study measures on real files.

What changes here: we treat the GGUF file itself as the measurement object. Not tok/s, not quality - the bytes and the promises, checked against the actual tensor types inside the file.

## The measurement object and metrics

Metrics:
- **file size** (bytes, MiB) per quant per model.
- **BPW** - bytes per weight, the ratio the quantizer reports (F16 = 16.00, Q8_0 ~8, Q4 ~4.3-5.5 depending on fallback).
- **general.file_type** (ftype) as written in the metadata: 1=F16, 8=Q8_0, 12=Q4_K, 2=Q4_0.
- **fallback count** - how many of the 272/352/216 tensors the quantizer reports as 'required fallback quantization', and which ones.
- **tensor type inventory** - read the GGUF back (via the runtime loader or gguf reader) and confirm the actual per-tensor types.

## Hypothesis board (decided before measurement)

Claims, committed before the files exist:

| # | Claim | Predicted |
|---|-------|-----------|
| C1 | A GGUF file's `general.file_type` is an accurate summary of its arithmetic | the metadata type and the dominant tensor types agree |
| C2 | Quantization delivers the advertised compression | Q8_0 ~ 8 BPW, Q4_0 ~ 4.5 BPW, Q4_K ~ 5 BPW, monotone across 135M/360M/1.7B |
| C3 | The quantizer coarsens tensors that do not fit the block | fallback count is 0 for Q8_0/Q4_0 (32-block divides 576/1536) and > 0 for Q4_K (256-block does not divide 576) |
| C4 | F16 conversion preserves the source | F16 GGUF loads with the same architecture metadata as the source config (block_count, embedding_length, head counts) |

## Protocol & constraints

- Models: SmolLM2-135M, 360M, 1.7B from the HF cache (already present, no downloads).
- Converter: `tools/convert_hf_to_gguf.py` + `tools/conversion/` vendored at llama.cpp commit `e3546c79` (the commit llama-cpp-python 0.3.34 bundles). `NO_LOCAL_GGUF=1` forces use of the pip `gguf` 0.19.0 (byte-identical to that commit's gguf-py).
- Output type: F16 first; then `llama_model_quantize` to Q8_0 (ftype 7), Q4_K_M (ftype 15), Q4_0 (ftype 2), `nthread=8`.
- Read-back: `llama_model_loader` metadata dump + tensor-type inventory.
- Constraint: the whole pipeline is offline; only CPU; no fp32 casts for the F16 path.

## The conversion rig

Run the vendored official converter (F16) for the three model dirs in the HF cache. Expected: 3 files in `gguf_cache/`. If the vocab path or other errors appear, note them - they are part of the artifact story.

In [20]:
import os, sys, subprocess, subprocess, collections

CONVERTER = 'tools/convert_hf_to_gguf.py'
os.environ['NO_LOCAL_GGUF'] = '1'

from llama_cpp import llama_model_quantize, llama_model_quantize_default_params
import gguf

FTYPE = {'q8_0': 7, 'q4_k_m': 15, 'q4_0': 2}
MODELS = ['135m', '360m', '1.7b']

In [21]:
REPO = r'C:\Users\Admin\OneDrive\Documents\Research_projects\pico_projects\gguf-inference'
os.chdir(REPO)
os.makedirs('gguf_cache', exist_ok=True)

HF = os.path.expanduser(r'~/.cache/huggingface/hub/models--HuggingFaceTB--SmolLM2-{m}/snapshots')

for tag in MODELS:
    m = tag.upper()
    snap =os.path.join(HF.format(m=m), os.listdir(HF.format(m=m))[0])
    out = f'gguf_cache/smollm2-{tag}-f16.gguf'
    if os.path.exists(out):
        print(f'[skip] {out}')
        continue
    cmd = [sys.executable, '-X', 'utf8', CONVERTER, snap, '--outfile', out, '--outtype', 'f16']
    r = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8')
    print(f'[convert] {tag}: rc={r.returncode} exists={os.path.exists(out)}')
    if r.returncode != 0:
        print(r.stderr[-500:])

[skip] gguf_cache/smollm2-135m-f16.gguf
[skip] gguf_cache/smollm2-360m-f16.gguf
[skip] gguf_cache/smollm2-1.7b-f16.gguf


## The quantization rig

Quantize each F16 file to Q8_0, Q4_K, Q4_0 via the runtime's own `llama_model_quantize`. Capture per-tensor logs. Record the 'N of M tensors required fallback quantization' line per quant type - this is the fine print.

In [22]:
for tag in MODELS:
    src = f'gguf_cache/smollm2-{tag}-f16.gguf'
    print(f'\n=== {tag} ===')
    for qname, ft in FTYPE.items():
        out = f'gguf_cache/smollm2-{tag}-{qname}.gguf'
        if os.path.exists(out):
            print(f'  [skip] {out}')
            continue
        p = llama_model_quantize_default_params()
        p.ftype = ft
        p.nthread = 8
        rc = llama_model_quantize(src.encode(), out.encode(), p)
        sz = os.path.getsize(out) / 1e6 if rc == 0 else -1
        print(f'  {qname:7s} ftype={ft:2d} rc={rc} size={sz:.1f} MB')
        if rc != 0:
            sys.exit(f'ERROR: quantize failed for {tag}/{qname}')


=== 135m ===
  [skip] gguf_cache/smollm2-135m-q8_0.gguf
  [skip] gguf_cache/smollm2-135m-q4_k_m.gguf
  [skip] gguf_cache/smollm2-135m-q4_0.gguf

=== 360m ===
  [skip] gguf_cache/smollm2-360m-q8_0.gguf
  [skip] gguf_cache/smollm2-360m-q4_k_m.gguf
  [skip] gguf_cache/smollm2-360m-q4_0.gguf

=== 1.7b ===
  [skip] gguf_cache/smollm2-1.7b-q8_0.gguf
  [skip] gguf_cache/smollm2-1.7b-q4_k_m.gguf
  [skip] gguf_cache/smollm2-1.7b-q4_0.gguf


## The file contract read-back

Read each GGUF back: file size, BPW, `general.file_type`, and the tensor-type inventory (how many f32/f16/q8_0/q4_k/q4_0 tensors actually landed inside). Compare the promise (file_type) against the reality (inventory).

In [23]:
files = []
for m in MODELS:
    for q in FTYPE:
        files.append(f'smollm2-{m}-{q}')

print(f'{"file":24s} {"ftype":5s} {"MiB":>7s} {"BPW":>5s}  tensor inventory')
print('-' * 100)
for f in files:
    p = f'gguf_cache/{f}.gguf'
    r = gguf.GGUFReader(p)
    ft = r.get_field('general.file_type').parts[-1][0]
    inv = collections.Counter(t.tensor_type.name for t in r.tensors)
    tot_e = sum(t.n_elements for t in r.tensors)
    tot_b = sum(t.n_bytes for t in r.tensors)
    bpw = tot_b * 8 / tot_e
    mib = os.path.getsize(p) / 1e6
    inv_s = ' '.join(f'{k}:{v}' for k, v in sorted(inv.items()))
    print(f'{f:24s} {ft:<5d} {mib:7.1f} {bpw:5.2f}  {inv_s}')

file                     ftype     MiB   BPW  tensor inventory
----------------------------------------------------------------------------------------------------
smollm2-135m-q8_0        7       144.8  8.51  F32:61 Q8_0:211
smollm2-135m-q4_k_m      15      105.5  6.17  F32:61 Q4_K:16 Q5_0:166 Q6_K:14 Q8_0:15
smollm2-135m-q4_0        2        91.7  5.35  F32:61 Q4_0:210 Q8_0:1
smollm2-360m-q8_0        7       386.4  8.50  F32:65 Q8_0:225
smollm2-360m-q4_k_m      15      270.6  5.94  F32:65 Q4_K:16 Q5_0:176 Q6_K:16 Q8_0:17
smollm2-360m-q4_0        2       229.1  5.03  F32:65 Q4_0:224 Q8_0:1
smollm2-1.7b-q8_0        7      1820.4  8.50  F32:49 Q8_0:169
smollm2-1.7b-q4_k_m      15     1055.6  4.93  F32:49 Q4_K:144 Q6_K:25
smollm2-1.7b-q4_0        2       990.7  4.62  F32:49 Q4_0:168 Q6_K:1


## Findings

| # | Claim | Predicted | Measured | Verdict |
|---|---|---|---|---|
| C1 | `general.file_type` is an accurate summary | metadata type and dominant tensor types agree | ftype=15 on 135M, but only 16/272 tensors are Q4_K; 166 are Q5_0, 14 Q6_K, 15 Q8_0 | ⚠️ Partial — truthful at block level, hides per-tensor fallback |
| C2 | Quantization delivers advertised compression | Q8_0 ~8 BPW, Q4_0 ~4.5 BPW, Q4_K ~5 BPW, monotone | Q8_0=8.51, Q4_0=5.35, Q4_K_M=6.17 (135M); 16 > 8.5 > 6.2 > 5.4 | ✅ Holds — BPW monotone; Q4_K_M not smallest on 135M/360M due to fallback |
| C3 | Quantizer coarsens tensors that don't fit the block | 0 fallback for 32-block, >0 for 256-block | Q8_0: 0; Q4_0: 1; Q4_K_M: 180 (135M), 192 (360M), 25 (1.7B) | ✅ Holds, stronger — divisibility predicts fallback exactly |
| C4 | F16 conversion preserves the source | F16 GGUF matches config.json | 135M: 30L/576H/9Q/3KV; 360M: 32/960/15/5; 1.7B: 24/2048/32/32 | ✅ Holds |

**Q4_K_M is a lie on 135M/360M, honest on 1.7B.** The file says `ftype=15` (Q4_K_M), but on 135M only 16 of 272 tensors are actually Q4_K — the rest fell back to Q5_0/Q6_K/Q8_0 because 576 is not divisible by 256. On 1.7B (width 2048, 2048/256 = 8), Q4_K_M is genuine: 144 Q4_K + 25 Q6_K, BPW 4.93.

**BPW is monotone within each model.** 16.00 → 8.51 → 6.17 → 5.35 (135M). But Q4_K_M is not the smallest on 135M/360M — fallback inflates it above Q4_0. On 1.7B, Q4_K_M (4.93) is close to Q4_0 (4.62) because there is no fallback.

**The embedding always falls back.** Every Q4_0 file has exactly 1 non-Q4_0 tensor (Q8_0 or Q6_K) — that is `token_embd.weight` (49152 × width). The width is divisible by 32 but the quantizer routes the embedding to Q8_0 anyway.

*(Every number in the Measured column is produced by the code cells above.)*

**Q4_K_M is a lie on 135M/360M, honest on 1.7B.** The file says ftype=15 (Q4_K_M), but on 135M only 16 of 272 tensors are actually Q4_K — the rest fell back to Q5_0/Q6_K/Q8_0 because 576 is not divisible by 256. On 1.7B (width 2048, 2048/256 = 8), Q4_K_M is genuine: 144 Q4_K + 25 Q6_K, BPW 4.93.

**BPW is monotone within each model.** 16.00 > 8.51 > 6.17 > 5.35 (135M). But Q4_K_M is not the smallest on 135M/360M — fallback inflates it above Q4_0. On 1.7B, Q4_K_M (4.93) is close to Q4_0 (4.62) because there is no fallback.

**The embedding always falls back.** Every Q4_0 file has exactly 1 non-Q4_0 tensor (Q8_0 or Q6_K) — that is token_embd.weight (49152 x width). The width is divisible by 32 but the quantizer routes the embedding to Q8_0 anyway.

*(Every number in the Measured column is produced by the code cells above.)*

## Discussion and verdict

**The claim in one line.** A GGUF file's name and even its `general.file_type` are promises; the tensor inventory is the truth. Q4_K_M — the type every model card advertises — is the type most likely to be silently replaced by a mix of coarser types, and the replacement is deterministic: it happens exactly when the tensor width is not divisible by 256.

**Why it matters.** This is the real-world instance of the scale-granularity finding from the earlier study. The earlier study showed that per-row scales beat whole-matrix scales because the block size determines how well the quantizer can represent the weights. This study shows the same principle at the file level: the 256-block of Q4_K_M is a promise the file keeps or breaks based on arithmetic, not intent. The 1.7B (width 2048) is the only SmolLM2 where Q4_K_M is clean. A deployer who reads the file header and assumes Q4_K_M is getting the advertised compression is misinformed on 135M and 360M — the real BPW is 6.17/5.94, not the ~4.5 they expected.

**Honesty note.** Three SmolLM2 sizes, one runtime (llama-cpp-python 0.3.34), simulated quantization via the official converter and quantizer. The margins are model-bound; the transferable content is the *divisibility rule* — a quant type T with block size B will fallback on any tensor whose width is not divisible by B — and the *verification ladder* (filename → metadata → tensor inventory → served output). The exact fallback counts (180/192/25) are specific to SmolLM2's widths; the rule is universal.

**Verdicts (measured):**

| # | Claim | Predicted | Measured | Verdict |
|---|---|---|---|---|
| C1 | file_type is accurate summary | metadata and dominant types agree | ftype=15 on 135M, only 16/272 Q4_K | Partial |
| C2 | Quantization delivers advertised compression | Q8_0~8, Q4_0~4.5, Q4_K~5 BPW, monotone | 16 > 8.5 > 6.2 > 5.4 (135M) | Holds |
| C3 | Quantizer coarsens tensors that don't fit | 0 fallback for 32-block, >0 for 256-block | Q8_0:0, Q4_0:1, Q4_K_M:180/192/25 | Holds, stronger |
| C4 | F16 conversion preserves source | F16 GGUF matches config.json | 30L/576H/9Q/3KV; 32/960/15/5; 24/2048/32/32 | Holds |

**The claim in one line.** A GGUF file name and even its general.file_type are promises; the tensor inventory is the truth. Q4_K_M is the type most likely to be silently replaced by coarser types, and the replacement is deterministic: it happens exactly when the tensor width is not divisible by 256.

**Why it matters.** This is the real-world instance of the scale-granularity finding from the earlier study. The 256-block of Q4_K_M is a promise the file keeps or breaks based on arithmetic, not intent. A deployer who reads the file header and assumes Q4_K_M is getting the advertised compression is misinformed on 135M and 360M — the real BPW is 6.17/5.94, not the ~4.5 they expected.

**Honesty note.** Three SmolLM2 sizes, one runtime (llama-cpp-python 0.3.34), simulated quantization via the official converter and quantizer. The margins are model-bound; the transferable content is the divisibility rule and the verification ladder. The exact fallback counts (180/192/25) are specific to SmolLM2 widths; the rule is universal.

## References

- llama.cpp / GGUF format docs (the vendored commit's `docs/gguf.md`).
- `gguf` pip package 0.19.0 (the converter library).
- llama-cpp-python 0.3.34 (the runtime that will serve these files in Studies 2-4).